In [ ]:
#importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#load dataset
df = pd.read_csv(r'c:\Users\User\OneDrive\Desktop\Marketing Campaign Effectiveness Analysis\digital_marketing_campaign_dataset.csv')

#display first 5 row
print(df.head())

In [ ]:
#Initial Exploration
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.info())
print(df.describe())
print(df.describe(include=['object','string']))
print(df.isnull().sum())

In [ ]:
#remove non useful columns
df.drop(columns=['AdvertisingPlatform','AdvertisingTool'],inplace=True)

In [ ]:
#check duplicates
print("Duplicate Rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)

In [ ]:
#Missing value treatment for numerical columns
num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

#missing value treatment of categorical value 
cat_cols = df.select_dtypes(include=["string", "object"]).columns
for cols in cat_cols:
    df[cols] = df[cols].fillna(df[cols].mode()[0])

In [ ]:
#outlier detection using box plot
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
for cols in num_cols:
    plt.figure()
    sns.boxplot(x=df[cols])
    plt.title(f'Boxplot of {cols}')
    plt.show()

In [ ]:
#Treatment of outlier using IQR method
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower,upper)

In [ ]:
#feature engineering
#conversion percentage
df["Conversion_Percentage"] = (df["ConversionRate"] * 10)

#CTR percentage
df["CTR_Percentage"] = (df["ClickThroughRate"] * 100)

#Email Engagement Rate
df["EmailEngagementRate"] = (df["EmailClicks"] / (df["EmailOpens"] + 1))

#visit quality score
df['VisitQualityScore'] = (df['PagesPerVisit'] * df['TimeOnSite'])

#Customer value Score
df["CustomerValueScore"] = (df['PreviousPurchases'] + (df['LoyaltyPoints']/1000))

#marketing efficiency
df["MarketingEfficiency"] = (df["ConversionRate"] * df["ClickThroughRate"])

#social engagement score
df["Engagement_Score"] = (df["SocialShares"] + df["EmailClicks"] + df["WebsiteVisits"])
df.head()

In [ ]:
#EDA
#Conversion by campaign channel
plt.figure(figsize=(8,5))
ax = sns.barplot(x='CampaignChannel',y='ConversionRate',data=df)
for container in ax.containers:
    ax.bar_label(container)
plt.title('Conversion Rate by Campaign Channel')
plt.show()

In [ ]:
#Distribution of Adspend
plt.figure(figsize=(8,5))
sns.histplot(x=df['AdSpend'], bins = 30, kde = True)
plt.title('Distribution of AdSpend')
plt.show()

In [ ]:
#campaign channel distribution
plt.figure(figsize=(8,5))
ax = sns.countplot(x ='CampaignChannel', data = df)
for container in ax.containers:
    ax.bar_label(container)
plt.title("Campaign Channel Distribution")
plt.show()

In [ ]:
#campaign type performance
plt.figure(figsize=(8,5))
ax = sns.barplot(x = 'CampaignType', y = 'ConversionRate', data = df)
for container in ax.containers:
    ax.bar_label(container)
    plt.title("Conversion Rate by Campaign Channel")
    plt.show()

In [ ]:
#Gender vs Conversion
plt.figure(figsize=(8,5))
ax = sns.barplot(x = 'Gender', y='ConversionRate', data = df)
for container in ax.containers:
    ax.bar_label(container)
    plt.title("Gender vs Conversion")
    plt.show()

In [ ]:
#AdSpend vs conversion rate
plt.figure(figsize=(8,5))
sns.scatterplot(x="AdSpend", y="ConversionRate", data=df)
plt.title("Adspend vs Conversion Rate")
plt.show()

In [ ]:
#website visit vs conversion
plt.figure(figsize=(8,5))
sns.boxplot(y='WebsiteVisits', x="Conversion", data=df)
plt.title("Website Visit vs Conversion")
plt.show()

In [ ]:
#correlation heatmap
corr = df.corr(numeric_only = True)
plt.figure(figsize=(14,8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt='.2f')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
#conversion funnel analysis
funnel = {
    "Visits":df["WebsiteVisits"].sum(),
    "Email Opens":df["EmailOpens"].sum(),
    "Email Clicks":df["EmailClicks"].sum(),
    "Conversions":df["Conversion"].sum()
}
funnel_df = pd.DataFrame(funnel.items(),columns=["Stage","Count"])
sns.barplot(x="Stage",y="Count",data=funnel_df)
plt.title("Marketing Funnel")
plt.show()

In [ ]:
#marketing funnel analysis
stages = ['Visited Website', 'Opened Email', 'Clicked Link', 'Converted']
counts = df[['WebsiteVisits', 'EmailOpens', 'EmailClicks', 'Conversion']].sum()

plt.figure(figsize=(8,5))
plt.bar(stages, counts)
plt.title('Marketing Funnel')
plt.xlabel('Stages')
plt.ylabel('Number of Users')
plt.show()

In [ ]:
#saved clean dataset
df.to_csv("Marketing_Campaign_Effectiveness_cleaned_Dataset.csv", index=False)